# RAG(Retrieval Augmented Generation) use local model, production ready RAG.


### 1. import the libraries and load the data

In [24]:
import os
import re
from dotenv import load_dotenv
load_dotenv()
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [25]:
def process_all_pdfs(data):
    """Process all PDF files in the specified directory."""
    all_documents = []
    pdf_dir = Path(data)

    # Find all PDF files in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing file:{pdf_file}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            
            all_documents.extend(documents)
            print(f"\n Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"\n Error: {e}")

    print(f"\n Total documents loaded:{len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("data")

Found 1 PDF files to process.

Processing file:data\Claude Certified Architect – Foundations Certification Exam Guide.pdf

 Loaded 40 pages

 Total documents loaded:40


In [26]:
all_pdf_documents

[Document(page_content=' \nClaude Certified Architect – Foundations Certification Exam Guide \nIntroduction \nThe Claude Certified Architect – Foundations certification validates that practitioners can make \ninformed decisions about tradeoffs when implementing real-world solutions with Claude. This \nexam tests foundational knowledge across Claude Code, the Claude Agent SDK, the Claude API, \nand Model Context Protocol (MCP) — the core technologies used to build production-grade \napplications with Claude. \n \nQuestions on this exam are grounded in realistic scenarios drawn from actual customer use \ncases, including building agentic systems for customer support, designing multi-agent research \npipelines, integrating Claude Code into CI/CD workflows, building developer productivity tools, \nand extracting structured data from unstructured documents. Candidates must demonstrate not \nonly conceptual knowledge but practical judgment about architecture, configuration, and \ntradeoffs i

In [50]:
# Chunking the documents using RecursiveCharacterTextSplitter from Langchain, it is a smarter variant of fixed_size chunking.It tries to cut at natural boundaries
def chunk_by_recursive_char(all_pdf_documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(all_pdf_documents)

    return split_docs

In [51]:
recursive_char = chunk_by_recursive_char(all_pdf_documents)
print(f"Split {len(all_pdf_documents)} documents into {len(recursive_char)} chunks")

for i, chunk in enumerate(recursive_char[:5]):
    print(f"--- Chunk {i} ---")
    print(f"Content: {chunk.page_content}\n")
    print(f"Metadata: {chunk.metadata}\n")
    

Split 40 documents into 115 chunks
--- Chunk 0 ---
Content: Claude Certified Architect – Foundations Certification Exam Guide 
Introduction 
The Claude Certified Architect – Foundations certification validates that practitioners can make 
informed decisions about tradeoffs when implementing real-world solutions with Claude. This 
exam tests foundational knowledge across Claude Code, the Claude Agent SDK, the Claude API, 
and Model Context Protocol (MCP) — the core technologies used to build production-grade 
applications with Claude. 
 
Questions on this exam are grounded in realistic scenarios drawn from actual customer use 
cases, including building agentic systems for customer support, designing multi-agent research 
pipelines, integrating Claude Code into CI/CD workflows, building developer productivity tools, 
and extracting structured data from unstructured documents. Candidates must demonstrate not 
only conceptual knowledge but practical judgment about architecture, configurati

In [52]:
# Fixed-size chunking by character count with overlap. This way the chunks may not make sense because it cuts in the middle of sentences.
def chunks_by_char(all_pdf_documents, chunk_size=1000, chunk_overlap=200):
    chunks = []

    for doc in all_pdf_documents:
            page = doc.page_content
            start_idx = 0

            while start_idx < len(page):
                end_idx = min(start_idx + chunk_size, len(page))
                chunk = page[start_idx:end_idx]
                chunks.append(chunk)

                start_idx = (
                end_idx - chunk_overlap if end_idx < len(page) else end_idx
             )

    return chunks   

In [55]:
all_pdf_chunks = chunks_by_char(all_pdf_documents)
print(f"Split {len(all_pdf_documents)} documents into {len(all_pdf_chunks)} chunks")
for i, chunk in enumerate(all_pdf_chunks[:5]):
    print(f"--- Chunk{i} ---\n{chunk}\n")

Split 40 documents into 115 chunks
--- Chunk0 ---
 
Claude Certified Architect – Foundations Certification Exam Guide 
Introduction 
The Claude Certified Architect – Foundations certification validates that practitioners can make 
informed decisions about tradeoffs when implementing real-world solutions with Claude. This 
exam tests foundational knowledge across Claude Code, the Claude Agent SDK, the Claude API, 
and Model Context Protocol (MCP) — the core technologies used to build production-grade 
applications with Claude. 
 
Questions on this exam are grounded in realistic scenarios drawn from actual customer use 
cases, including building agentic systems for customer support, designing multi-agent research 
pipelines, integrating Claude Code into CI/CD workflows, building developer productivity tools, 
and extracting structured data from unstructured documents. Candidates must demonstrate not 
only conceptual knowledge but practical judgment about architecture, configuration, and 

In [6]:
def chunks_by_sentences(all_pdf_documents, max_sentences_per_chunk=5, overlap_sentences=1):
    chunks = []

    for doc in all_pdf_documents:
        page = doc.page_content
        sentences = re.split(r"(?<=[.?!])\s+",page)

        start_idx = 0
        while start_idx < len(sentences):
            end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
            current_chunk = sentences[start_idx:end_idx]
            chunks.append(" ".join(current_chunk))
            start_idx = end_idx - overlap_sentences if end_idx < len(sentences) else end_idx

    return chunks


In [32]:
all_pdf_sentence_chunks = chunks_by_sentences(all_pdf_documents)
print(f"Total sentence-based chunks created: {len(all_pdf_sentence_chunks)}")
for i, chunk in enumerate(all_pdf_sentence_chunks[:5]):
    print(f"--- Sentence Chunk {i} ---\n{chunk}\n")

Total sentence-based chunks created: 85
--- Sentence Chunk 0 ---
 
Claude Certified Architect – Foundations Certification Exam Guide 
Introduction 
The Claude Certified Architect – Foundations certification validates that practitioners can make 
informed decisions about tradeoffs when implementing real-world solutions with Claude. This 
exam tests foundational knowledge across Claude Code, the Claude Agent SDK, the Claude API, 
and Model Context Protocol (MCP) — the core technologies used to build production-grade 
applications with Claude. Questions on this exam are grounded in realistic scenarios drawn from actual customer use 
cases, including building agentic systems for customer support, designing multi-agent research 
pipelines, integrating Claude Code into CI/CD workflows, building developer productivity tools, 
and extracting structured data from unstructured documents. Candidates must demonstrate not 
only conceptual knowledge but practical judgment about architecture, configu

In [ ]:
def chunk_by_section(all_pdf_documents):
    chunks = []

    for doc in all_pdf_documents:
        page = doc.page_content
        pattern = r"\n## "
        chunks.extend(re.split(pattern,page))

    return chunks
    

In [164]:
chunk_section = chunk_by_section(all_pdf_documents)
print(len(chunk_section))
print(chunk_section[2])

41
Introduction
A skill is a set of instructions - packaged as a simple folder - that teaches Claude 
how to handle specific tasks or workflows. Skills are one of the most powerful 
ways to customize Claude for your specific needs. Instead of re-explaining your 
preferences, processes, and domain expertise in every conversation, skills let you 
teach Claude once and benefit every time. 
Skills are powerful when you have repeatable workflows: generating frontend 
designs from specs, conducting research with consistent methodology, creating 
documents that follow your team's style guide, or orchestrating multi-step 
processes. They work well with Claude's built-in capabilities like code execution 
and document creation. For those building MCP integrations, skills add another 
powerful layer helping turn raw tool access into reliable, optimized workflows.
This guide covers everything you need to know to build effective skills - from 
planning and structure to testing and distribution. Whe